# Climate as Data

In this notebook, we'll learn to fetch real climate data from NOAA, understand it as numbers measured over time, and discover patterns through visualization.

Just as audio is thousands of amplitude samples per second, **climate data is thousands of temperature measurements over years**. Same idea, different domain.

## Step 1: Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Step 2: What Is Climate Data?

A weather station records temperature every day. One number per day, for decades. That gives us a **time series** — a list of numbers ordered by time.

| Date | TMAX | TMIN | TAVG |
|------|------|------|------|
| 2000-01-01 | 27.2 | 17.2 | 22.2 |
| 2000-01-02 | 26.1 | 18.9 | 22.8 |
| ... | ... | ... | ... |

We'll fetch ~9,000 daily records from a single Miami weather station spanning 25 years.

## Step 3: Fetch Data from NOAA

In [ ]:
url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "daily-summaries",
    "stations": "USW00012839",       # Miami International Airport
    "startDate": "2000-01-01",
    "endDate": "2024-12-31",
    "dataTypes": "TMAX,TMIN,TAVG",
    "units": "metric",
    "format": "json"
}

print("Fetching data from NOAA...")
r = requests.get(url, params=params, timeout=60)
print("Status:", r.status_code)

data = r.json()
print("Records fetched:", len(data))

In [ ]:
df = pd.DataFrame(data)
df.head()

## Step 4: Clean the Data

In [ ]:
# Convert DATE to datetime
df["DATE"] = pd.to_datetime(df["DATE"])

# Convert temperature columns to numeric
for col in ["TAVG", "TMAX", "TMIN"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing TAVG with the average of TMAX and TMIN
df["TAVG"] = df["TAVG"].fillna((df["TMAX"] + df["TMIN"]) / 2)

# Add useful time columns
df["year"] = df["DATE"].dt.year
df["month"] = df["DATE"].dt.month
df["dayofyear"] = df["DATE"].dt.dayofyear

print(f"Date range: {df['DATE'].min().date()} to {df['DATE'].max().date()}")
print(f"Records: {len(df)}")
print(f"Missing TAVG after fill: {df['TAVG'].isna().sum()}")
df.describe()

## Step 5: The Climate "Waveform" — Daily Temperature

Just like an audio waveform shows amplitude over time, a temperature time series shows temperature over time. It tells us **when** temperatures change but not the deeper patterns.

In [ ]:
# Full 25-year daily time series
plt.figure(figsize=(14, 5))
plt.plot(df["DATE"], df["TAVG"], linewidth=0.3, color="steelblue")
plt.title("Miami Daily Average Temperature (2000–2024)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Zoom into a single year
df_2010 = df[df["year"] == 2010]

plt.figure(figsize=(12, 5))
plt.plot(df_2010["DATE"], df_2010["TAVG"], color="steelblue")
plt.fill_between(df_2010["DATE"], df_2010["TMIN"], df_2010["TMAX"], alpha=0.2, color="steelblue")
plt.title("Miami Daily Temperature (2010) — with TMIN/TMAX range")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.grid(True, alpha=0.3)
plt.show()

## Step 6: Annual Averages and Trends

Group by year and compute the mean to reveal the long-term trend.

In [ ]:
annual_avg = df.groupby("year")["TAVG"].mean()

# Linear regression
x = annual_avg.index.values
y = annual_avg.values
coeffs = np.polyfit(x, y, 1)
trend = np.poly1d(coeffs)

plt.figure(figsize=(10, 5))
plt.plot(x, y, "o-", color="steelblue", label="Annual Average")
plt.plot(x, trend(x), "r--", linewidth=2, label="Trend Line")
plt.title("Miami Annual Average Temperature with Trend")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

warming_per_decade = coeffs[0] * 10
print(f"Trend: {coeffs[0]:.4f} °C/year = {warming_per_decade:.2f} °C/decade")

## Step 7: The Climate "Spectrogram" — Temperature Heatmap

Just as a **spectrogram** reveals frequency patterns hidden in an audio waveform, a **heatmap** reveals year-to-year patterns hidden in a temperature time series.

- **X-axis**: Day of year (1–365)
- **Y-axis**: Year (2000–2024)
- **Color**: Temperature (cool = blue, warm = red)

In [ ]:
pivot = df.pivot_table(index="year", columns="dayofyear", values="TAVG")

plt.figure(figsize=(15, 7))
sns.heatmap(pivot, cmap="coolwarm", cbar_kws={"label": "°C"},
            xticklabels=30, yticklabels=1)
plt.title("Miami Daily Temperature Heatmap (2000–2024)")
plt.xlabel("Day of Year")
plt.ylabel("Year")
plt.show()

print("Each row is a year. Each column is a day.")
print("You can see the seasonal cycle repeating every row.")
print("Look for years that are warmer (more red) or cooler (more blue) overall.")

## Step 8: Monthly Climatology

The average temperature for each month, computed across all 25 years. This is the **average seasonal shape** of Miami's climate.

In [ ]:
monthly_clim = df.groupby("month")["TAVG"].mean()
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

plt.figure(figsize=(8, 5))
plt.bar(range(1, 13), monthly_clim.values, color="coral")
plt.xticks(range(1, 13), month_names)
plt.title("Miami Monthly Climatology (2000–2024)")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(axis="y", alpha=0.3)
plt.show()

## Step 9: Anomalies — Deviation from Normal

An **anomaly** is how far a measurement deviates from the long-term average. Positive anomaly = warmer than normal. Negative = cooler.

In [ ]:
# Compute monthly baseline (long-term mean for each month)
monthly_baseline = df.groupby("month")["TAVG"].transform("mean")
df["anomaly"] = df["TAVG"] - monthly_baseline

# Annual mean anomaly
annual_anomaly = df.groupby("year")["anomaly"].mean()

colors = ["#e74c3c" if v >= 0 else "#3498db" for v in annual_anomaly.values]

plt.figure(figsize=(10, 5))
plt.bar(annual_anomaly.index, annual_anomaly.values, color=colors)
plt.title("Miami Annual Temperature Anomaly (2000–2024)")
plt.xlabel("Year")
plt.ylabel("Anomaly (°C)")
plt.axhline(0, color="black", linewidth=0.8)
plt.grid(axis="y", alpha=0.3)
plt.show()

print("Red bars = warmer than average. Blue bars = cooler than average.")

## Step 10: Rolling Mean — Smoothing the Signal

A **rolling mean** averages nearby years to smooth out noise and reveal the underlying trend.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(annual_avg.index, annual_avg.values, "o-", alpha=0.5, label="Annual Average")
plt.plot(annual_avg.index, annual_avg.rolling(5, center=True).mean(),
         "r-", linewidth=2.5, label="5-Year Rolling Mean")
plt.title("Miami Annual Average with 5-Year Smoothing")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## What's Next

In **Notebook 2**, we'll explore and compare **5 US cities** with very different climates — building visual intuition for tropical, continental, Mediterranean, and marine climate types.